# Домашнее задание: Мультиклассовая классификация аудио с помощью PyTorch

На семинаре мы реализовали линейный слой и SGD вручную для бинарной классификации цифр `0` и `9`. Теперь ваша задача – **использовать готовые модули PyTorch** для построения и обучения модели, способной различать **все 10 цифр** (0-9).

**Что нужно сделать:**
1. Загрузить полный датасет **AudioMNIST** (все 10 классов).
2. Создать `Dataset` и `DataLoader`.
3. Построить нейросеть с помощью `nn.Linear` и `nn.ReLU`.
4. Обучить модель с помощью `nn.CrossEntropyLoss` и `optim.SGD`.
5. Оценить качество на валидационной выборке.
6. Визуализировать динамику обучения.
7. Вычисление мел-спектрограмм
8. Обучение модели на мел-спектрограммах

В ячейках ниже приведён подробный код и пояснения. Некоторые участки могут быть оставлены для самостоятельной реализации – при необходимости вы сможете дополнить их.

## 1. Подготовка окружения и загрузка данных

Импортируем необходимые библиотеки. Если вы работаете в Google Colab, раскомментируйте строчку с установкой `gdown`.

In [ ]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# Фиксируем случайность для воспроизводимости
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# При необходимости установите gdown
# !pip install gdown -q

Скачаем и распакуем AudioMNIST. Архив содержит 60 папок с записями дикторов.

In [ ]:
import gdown
import zipfile

FILE_ID = "13E8JVJdNYS9K9ubdGcruLSMLUD10OLbx"
output = "audio_mnist.zip"

if not os.path.exists(output):
    gdown.download(id=FILE_ID, output=output, quiet=False)

# Распаковка
with zipfile.ZipFile(output, 'r') as zip_ref:
    zip_ref.extractall(".")

## 2. Реализация `torch.utils.data.Dataset`

Создадим класс `AudioMNISTDataset`. Он должен:
- Сканировать все `.wav` файлы в папках `00` – `59`.
- Извлекать метку из имени файла (первый символ до `_`).
- Приводить аудио к фиксированной длине `16000` (padding / truncation).
- Нормализовать сигнал в диапазон `[-1, 1]`.
- Возвращать тензор звука и целочисленную метку.

In [ ]:
class AudioMNISTDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, target_len=16000):
        self.root_dir = root_dir
        self.target_len = target_len
        self.files = []
        self.labels = []

        speaker_dirs = [d for d in glob.glob(os.path.join(root_dir, "*")) if os.path.isdir(d)]
        for speaker_dir in speaker_dirs:
            wav_paths = glob.glob(os.path.join(speaker_dir, "*.wav"))
            for wav_path in wav_paths:
                basename = os.path.basename(wav_path)
                label_str = basename.split("_")[0]
                if label_str.isdigit():
                    self.files.append(wav_path)
                    self.labels.append(int(label_str))

        # Сортируем для воспроизводимости
        sorted_pairs = sorted(zip(self.labels, self.files))
        self.labels, self.files = [list(x) for x in zip(*sorted_pairs)]
        print(f"Найдено {len(self.files)} аудиофайлов")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        wav_path = self.files[idx]
        label = self.labels[idx]

        #TODO: Реализуйте весь ход обработки и загрузки аудио

        waveform = None

        return waveform.float(), torch.tensor(label, dtype=torch.long)

In [ ]:
# Создаём полный датасет
full_dataset = AudioMNISTDataset('data', target_len=16000)

Разобьём данные на обучающую (80%) и валидационную (20%) выборки.

In [ ]:
train_ratio = 0.8
train_size = int(len(full_dataset) * train_ratio)
val_size = len(full_dataset) - train_size

train_ds, val_ds = torch.utils.data.random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

Создадим `DataLoader` для пакетной обработки.

In [ ]:
BATCH_SIZE = 64
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

## 3. Построение модели с использованием `nn.Linear` и `nn.ReLU`

### Математическое описание слоёв

**Линейный слой (Fully Connected):**

$$
\mathbf{y} = \mathbf{x} \mathbf{W}^\top + \mathbf{b}
$$

- $\mathbf{x} \in \mathbb{R}^{B \times D_{in}}$ – входной батч,
- $\mathbf{W} \in \mathbb{R}^{D_{out} \times D_{in}}$ – матрица весов,
- $\mathbf{b} \in \mathbb{R}^{D_{out}}$ – смещение,
- $\mathbf{y} \in \mathbb{R}^{B \times D_{out}}$ – выход.

В PyTorch он реализован классом `nn.Linear(D_in, D_out)`.

**ReLU (Rectified Linear Unit):**

$$
\text{ReLU}(z) = \max(0, z)
$$

Нелинейность, применяемая поэлементно. Ускоряет сходимость и помогает избежать проблемы исчезающих градиентов. В PyTorch: `nn.ReLU()`.

### Архитектура модели

Мы построим двухслойный перцептрон:

1. **Входной слой:** 16000 (длина аудио) → `hidden_dim` (например, 128)
2. **ReLU**
3. **Выходной слой:** `hidden_dim` → 10 (число классов)

Выход модели – **логиты** (не нормализованные), которые затем передаются в функцию потерь `CrossEntropyLoss`.

Реализовывать свою функцию `backward` не нужно - **PyTorch** сделает все автоматически!


In [ ]:
class AudioMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        pass

    def forward(self, x):
        pass

In [ ]:
INPUT_DIM = 16000
HIDDEN_DIM = 128
NUM_CLASSES = 10

model = AudioMLP(INPUT_DIM, HIDDEN_DIM, NUM_CLASSES)
print(model)

## 4. Функция потерь и оптимизатор

### Кросс-энтропия для мультиклассовой классификации

Для задачи с $C$ классами используется **Cross-Entropy Loss**:

$$
\mathcal{L} = -\frac{1}{B} \sum_{i=1}^{B} \log\left( \frac{ e^{z_{i, y_i}} }{ \sum_{c=1}^{C} e^{z_{i, c}} } \right)
$$

где $z_{i, c}$ – логит для $i$-го примера и класса $c$, $y_i$ – истинная метка.

В PyTorch класс `nn.CrossEntropyLoss` объединяет `LogSoftmax` и `NLLLoss`, поэтому на вход ожидает **не нормализованные логиты**.

### Стохастический градиентный спуск (SGD)

Обновление параметров на каждом шаге:

$$
\theta \leftarrow \theta - \eta \cdot \nabla_\theta \mathcal{L}
$$

где $\eta$ – скорость обучения (`lr`). В PyTorch: `optim.SGD(params, lr=...)`.

In [ ]:
criterion = None
optimizer = None

## Метрики качества мультиклассовой классификации

Помимо обычной доли правильных ответов (**Accuracy**), для несбалансированных классов важны метрики, учитывающие ошибки первого и второго рода: **Precision**, **Recall** и их гармоническое среднее – **F1-score**.

### Определения

Пусть $C$ – число классов. Для каждого класса $c \in \{1, \dots, C\}$:

- **True Positives (TP$_{c}$)** – количество примеров, правильно отнесённых к классу $c$.
- **False Positives (FP$_{c}$)** – количество примеров, ошибочно отнесённых к классу $c$.
- **False Negatives (FN$_{c}$)** – количество примеров класса $c$, ошибочно отнесённых к другим классам.

Тогда:

$$
\text{Precision}_c = \frac{\text{TP}_c}{\text{TP}_c + \text{FP}_c}, \qquad
\text{Recall}_c = \frac{\text{TP}_c}{\text{TP}_c + \text{FN}_c}, \qquad
\text{F1}_c = 2 \cdot \frac{\text{Precision}_c \cdot \text{Recall}_c}{\text{Precision}_c + \text{Recall}_c}
$$

Чтобы получить единую метрику по всем классам, используют усреднение:

- **Micro-averaging**: суммируем TP, FP, FN по всем классам и вычисляем метрики по суммарным значениям. Эквивалентно взвешенному усреднению с весами, пропорциональными поддержке классов.
- **Macro-averaging**: вычисляем метрику для каждого класса независимо и берём среднее арифметическое (все классы равнозначны).
- **Weighted-averaging**: среднее арифметическое, взвешенное по количеству истинных примеров каждого класса.

### Вычисление в PyTorch

Основная идея – построить **матрицу ошибок (confusion matrix)** размера $C \times C$, где элемент $(i, j)$ – количество примеров класса $i$, предсказанных как класс $j$. Из неё легко получить TP, FP, FN.

Вам предлагается реализовать функции:

1. `confusion_matrix(y_true, y_pred, num_classes)` – возвращает тензор `(num_classes, num_classes)`.
2. `calculate_accuracy(y_true, y_pred)` – доля правильных ответов.
3. `calculate_precision_recall_f1(y_true, y_pred, average='macro')` – возвращает precision, recall, f1 для заданного типа усреднения (`'micro'`, `'macro'`, `'weighted'`).

Все операции должны быть векторными, без циклов по классам (допустимо использование `torch.bincount` или `torch.histogram`).

Ниже заготовки функций. Дополните их кодом, чтобы они проходили проверочные assert'ы.

In [ ]:
def confusion_matrix(y_true, y_pred, num_classes):
    """
    Строит матрицу ошибок размера (num_classes, num_classes) без циклов.

    Аргументы:
        y_true: тензор истинных меток, shape (N,)
        y_pred: тензор предсказанных меток, shape (N,)
        num_classes: количество классов

    Возвращает:
        cm: тензор (num_classes, num_classes), где cm[i, j] – число примеров класса i,
            предсказанных как класс j.
    """
    # Убедимся, что тензоры одномерные и целочисленные
    y_true = y_true.long().view(-1)
    y_pred = y_pred.long().view(-1)

    # TODO: Реализуйте функцию, ниже подсказки
    # Формируем линейный индекс: i * num_classes + j
    # Это позволит использовать bincount для построения матрицы одним вызовом
    pass

def calculate_accuracy(y_true, y_pred):
    """
    Вычисляет долю правильных ответов (accuracy).
    """
    pass

def calculate_precision_recall_f1(y_true, y_pred):
    """
    Вычисляет precision, recall, f1-score для мультиклассовой классификации.

    Аргументы:
        y_true: тензор истинных меток (N,)
        y_pred: тензор предсказанных меток (N,)

    Возвращает:
        precision, recall, f1: числа (float)
    """
    y_true = y_true.long().view(-1)
    y_pred = y_pred.long().view(-1)
    num_classes = int(max(y_true.max().item(), y_pred.max().item())) + 1

    # Строим матрицу ошибок
    cm = confusion_matrix(y_true, y_pred, num_classes).float()

    # Диагональные элементы – TP для каждого класса
    tp = cm.diag()
    # Сумма по столбцам – все предсказания класса c (TP + FP)
    pred_per_class = cm.sum(dim=0)
    # Сумма по строкам – все истинные примеры класса c (TP + FN)
    true_per_class = cm.sum(dim=1)

    # Для избежания деления на ноль добавляем маленькое число (eps)
    eps = 1e-10

    # Вычисляем метрики для каждого класса и усредняем арифметически
    # TODO: Реализуйте вычисление f1 и других метрик.
    precision = 0
    recall = 0
    f1 = 0

    return precision, recall, f1

### Проверка корректности

Выполните следующую ячейку. Если все assert'ы проходят молча, ваша реализация верна.

In [ ]:
# Пример для проверки
y_true = torch.tensor([0, 1, 2, 2, 1, 0])
y_pred = torch.tensor([0, 2, 2, 2, 1, 1])
num_classes = 3

cm = confusion_matrix(y_true, y_pred, num_classes)
expected_cm = torch.tensor([
    [1, 1, 0],
    [0, 1, 1],
    [0, 0, 2]
], dtype=torch.long)
assert torch.all(cm == expected_cm), f"Матрица ошибок неверна: {cm}"

acc = calculate_accuracy(y_true, y_pred)
assert abs(acc - 4/6) < 1e-6, f"Accuracy должно быть 4/6, получено {acc}"

prec_macro, rec_macro, f1_macro = calculate_precision_recall_f1(y_true, y_pred)
# Класс 0: TP=1, FP=0, FN=1 -> prec=1, rec=0.5, f1=0.666...
# Класс 1: TP=1, FP=1, FN=1 -> prec=0.5, rec=0.5, f1=0.5
# Класс 2: TP=2, FP=1, FN=0 -> prec=2/3=0.666..., rec=1, f1=0.8
# Macro: prec = (1+0.5+0.666...)/3 ≈ 0.7222, rec = (0.5+0.5+1)/3 = 0.666..., f1 ≈ (0.666+0.5+0.8)/3 = 0.6555
assert abs(prec_macro - 0.7222222) < 1e-4, f"Macro precision: {prec_macro}"
assert abs(rec_macro - 2/3) < 1e-4, f"Macro recall: {rec_macro}"
assert abs(f1_macro - 0.655555) < 1e-4, f"Macro f1: {f1_macro}"

print("Все проверки пройдены!")

## 5. Цикл обучения

Будем обучать модель несколько эпох. На каждой эпохе:
- Проходим по обучающему `DataLoader`:
  - Обнуляем градиенты (`optimizer.zero_grad()`)
  - Делаем прямой проход
  - Вычисляем loss
  - Делаем обратный проход (`loss.backward()`)
  - Шаг оптимизатора (`optimizer.step()`)
- Вычисляем loss и точность на валидации (без вычисления градиентов).
- Сохраняем метрики в `history` для последующей визуализации.

In [ ]:
EPOCHS = 10
history = {
    "train_loss": [], "val_loss": [],
    "train_acc": [], "val_acc": [],
    "val_precision": [], "val_recall": [], "val_f1": []
}

for epoch in range(EPOCHS):
    # ----- Обучение -----
    model.train()
    train_loss_sum, train_correct, train_total = 0.0, 0, 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1} | Train", leave=False)
    for x_batch, y_batch in pbar:
        x_batch, y_batch = x_batch, y_batch

        #TODO: Реализуйте шаг обучения с использованием готовый модулей Torch

        train_loss_sum += loss.item() * x_batch.size(0)
        preds = torch.argmax(logits, dim=1)
        train_correct += (preds == y_batch).sum().item()
        train_total += x_batch.size(0)

    train_loss = train_loss_sum / train_total
    train_acc = train_correct / train_total

    # ----- Валидация -----
    model.eval()
    val_loss_sum, val_correct, val_total = 0.0, 0, 0
    all_val_preds = []
    all_val_labels = []

    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            #TODO: Реализуйте шаг предсказания класса и подсчета функции ошибки с использованием готовый модулей Torch

            val_loss_sum += loss.item() * x_batch.size(0)
            preds = torch.argmax(logits, dim=1)
            val_correct += (preds == y_batch).sum().item()
            val_total += x_batch.size(0)

            # Сохраняем предсказания и метки для вычисления Precision/Recall/F1
            all_val_preds.append(preds.cpu())
            all_val_labels.append(y_batch.cpu())

    val_loss = val_loss_sum / val_total
    val_acc = val_correct / val_total

    # Объединяем все предсказания и метки валидации
    val_preds_tensor = torch.cat(all_val_preds)
    val_labels_tensor = torch.cat(all_val_labels)

    # Вычисляем метрики (macro average)
    val_precision, val_recall, val_f1 = calculate_precision_recall_f1(
        val_labels_tensor, val_preds_tensor
    )

    # Сохраняем историю
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["val_precision"].append(val_precision)
    history["val_recall"].append(val_recall)
    history["val_f1"].append(val_f1)

    print(f"Epoch {epoch+1:02d} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.2%} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.2%} | "
          f"Val F1: {val_f1:.4f}")

## 6. Визуализация результатов

Построим графики функции потерь и точности на обучающей и валидационной выборках.

In [ ]:
plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
plt.plot(history["train_loss"], marker="o", label="Train Loss")
plt.plot(history["val_loss"], marker="s", label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.legend()
plt.title("Loss")

plt.subplot(2, 2, 2)
plt.plot(history["train_acc"], marker="o", label="Train Acc")
plt.plot(history["val_acc"], marker="s", label="Val Acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.grid(True, alpha=0.3)
plt.legend()
plt.title("Accuracy")

plt.subplot(2, 2, 3)
plt.plot(history["val_f1"], marker="o", color="green", label="Val F1 (macro)")
plt.xlabel("Epoch")
plt.ylabel("F1-score")
plt.grid(True, alpha=0.3)
plt.legend()
plt.title("F1 Score (Macro)")

plt.tight_layout()
plt.show()

## 7. Использование мел-спектрограмм в качестве признаков

До сих пор мы подавали в нейросеть сырые отсчёты звукового сигнала (16 000 чисел на пример). Однако слух человека воспринимает звук нелинейно по частоте, и для задач классификации речи/аудио гораздо эффективнее использовать частотно-временные представления, например **мел-спектрограммы**.

### Что такое мел-спектрограмма?

1. **Спектрограмма** – это представление сигнала в координатах «время–частота», полученное с помощью оконного преобразования Фурье (STFT).
2. **Мел-шкала** – психоакустическая шкала, моделирующая восприятие высоты звука человеком. Низкие частоты разрешаются лучше, чем высокие.
3. **Мел-спектрограмма** получается применением набора треугольных фильтров (mel filter banks) к энергетическому спектру STFT, после чего обычно берётся логарифм амплитуд.

В PyTorch готовый трансформ `torchaudio.transforms.MelSpectrogram` выполняет все эти шаги.

### Изменение подхода

Мы **не будем** модифицировать класс `Dataset`. Вместо этого будем преобразовывать каждый батч **«на лету» внутри цикла обучения**.

Вам нужно:
1. Создать экземпляр `MelSpectrogram` с подходящими параметрами.
2. В цикле обучения перед подачей в модель применить преобразование к тензору `x_batch`.
3. Построить новую модель, учитывая новые размеры входа (количество mel-фильтров × количество временных окон).
4. Обучить и сравнить результаты с моделью на сырых данных.

In [ ]:
import torchaudio.transforms as T

# Параметры мел-спектрограммы
sample_rate = 16000
n_fft = 400          # размер окна FFT (25 мс при 16 кГц)
hop_length = 160     # шаг между окнами (10 мс)
n_mels = 64          # количество мел-фильтров

mel_transform = None

Определим размер входа для новой модели. Для одного примера длиной 16 000 отсчётов количество временных окон вычисляется как:
`time_frames = (16000 - n_fft) // hop_length + 1`.

При `n_fft=400` и `hop_length=160` получаем `(16000 - 400)//160 + 1 = 99`.
Таким образом, каждый пример будет иметь форму `(n_mels=64, time_frames=99)`, т.е. всего `64 * 99 = 6336` признаков.

In [ ]:
# Проверим размерность на одном батче
dummy_batch = torch.randn(4, 16000)
mel_batch = mel_transform(dummy_batch)
print(f"Форма после преобразования: {mel_batch.shape}")  # ожидаем (4, 64, 101)
input_dim_mel = mel_batch.shape[1] * mel_batch.shape[2]  # 6464
print(f"Размер плоского вектора признаков: {input_dim_mel}")

### Новая модель

Создадим MLP с тем же числом скрытых нейронов, но с входным размером `6336`.

In [ ]:
model_mel = AudioMLP(input_dim=input_dim_mel, hidden_dim=128, num_classes=10)
criterion_mel = None
optimizer_mel = None

### Цикл обучения с мел-спектрограммами

Обратите внимание: Мы также выпрямляем (flatten) полученный тензор перед подачей в полносвязный слой.

In [ ]:
EPOCHS = 10
history_mel = {
    "train_loss": [], "val_loss": [],
    "train_acc": [], "val_acc": [],
    "val_f1": []
}

for epoch in range(EPOCHS):
    # ----- Обучение -----
    model_mel.train()
    train_loss_sum, train_correct, train_total = 0.0, 0, 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1} | Train", leave=False)
    for x_batch, y_batch in pbar:
        x_batch, y_batch = x_batch, y_batch

        # Преобразование в мел-спектрограммы
        mel = mel_transform(x_batch)                    # (B, n_mels, T)
        features = mel.flatten(start_dim=1)          # (B, n_mels*T)

        features = (features - features.mean()) / features.std() # Нормирование мел-спектрограммы для стабильности обучения

        #TODO: Реализуйте шаг обучения с использованием готовый модулей Torch

        train_loss_sum += loss.item() * x_batch.size(0)
        preds = torch.argmax(logits, dim=1)
        train_correct += (preds == y_batch).sum().item()
        train_total += x_batch.size(0)

    train_loss = train_loss_sum / train_total
    train_acc = train_correct / train_total

    # ----- Валидация -----
    model_mel.eval()
    val_loss_sum, val_correct, val_total = 0.0, 0, 0
    all_val_preds = []
    all_val_labels = []

    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            mel = mel_transform(x_batch)
            features = mel.flatten(start_dim=1)

            features = (features - features.mean()) / features.std()

            #TODO: Реализуйте шаг предсказания класса и подсчета функции ошибки с использованием готовый модулей Torch

            val_loss_sum += loss.item() * x_batch.size(0)
            preds = torch.argmax(logits, dim=1)
            val_correct += (preds == y_batch).sum().item()
            val_total += x_batch.size(0)

            all_val_preds.append(preds.cpu())
            all_val_labels.append(y_batch.cpu())

    val_loss = val_loss_sum / val_total
    val_acc = val_correct / val_total

    val_preds_tensor = torch.cat(all_val_preds)
    val_labels_tensor = torch.cat(all_val_labels)
    _, _, val_f1 = calculate_precision_recall_f1(val_labels_tensor, val_preds_tensor)

    history_mel["train_loss"].append(train_loss)
    history_mel["val_loss"].append(val_loss)
    history_mel["train_acc"].append(train_acc)
    history_mel["val_acc"].append(val_acc)
    history_mel["val_f1"].append(val_f1)

    print(f"Epoch {epoch+1:02d} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.2%} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.2%} | "
          f"Val F1: {val_f1:.4f}")

In [ ]:
plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
plt.plot(history_mel["train_loss"], marker="o", label="Train Loss")
plt.plot(history_mel["val_loss"], marker="s", label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.legend()
plt.title("Loss")

plt.subplot(2, 2, 2)
plt.plot(history_mel["train_acc"], marker="o", label="Train Acc")
plt.plot(history_mel["val_acc"], marker="s", label="Val Acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.grid(True, alpha=0.3)
plt.legend()
plt.title("Accuracy")

plt.subplot(2, 2, 3)
plt.plot(history_mel["val_f1"], marker="o", color="green", label="Val F1 (macro)")
plt.xlabel("Epoch")
plt.ylabel("F1-score")
plt.grid(True, alpha=0.3)
plt.legend()
plt.title("F1 Score (Macro)")

plt.tight_layout()
plt.show()